# Part 3 — CycleGAN: Monet ↔ Photo

This notebook trains a CycleGAN from scratch for unpaired image translation. Domain A contains Monet paintings and domain B contains photographs. `G_A2B` translates Monet images into photos, while `G_B2A` translates photos into Monet-style images.

The recorded run uses 256 × 256 images, seed 3143, 200 epochs, 512 update pairs per epoch, and the complete sampling pools of 300 Monet images and 7,038 photos. Checkpoints are compared with a fixed 300-photo subset, and epoch 190 is the selected checkpoint for the published result.

The notebook keeps training, evaluation, checkpoint selection, full-photo inference, plots, metrics, audit preparation, and submission generation in one reproducible workflow.

## 0. Running the notebook

Place this notebook in a writable project folder with the following structure:

```text
Part3.ipynb
data/
  monet_jpg/
  photo_jpg/
  real_stats.npz
  sample_submission.csv   # optional until submission generation
```

Run the setup, data validation, evaluator check, and benchmark cells first. Review their results before enabling the full training cell. A long execution can be run from a persistent terminal:

```bash
jupyter nbconvert --to notebook --execute Part3.ipynb   --ExecutePreprocessor.timeout=-1   --output Part3_executed.ipynb
```

Download the executed notebook, selected checkpoint, logs, and output folder after the run. Relative paths are resolved from the notebook's working directory; change `ROOT` only when the data are stored elsewhere.

### 0.1 Dependencies

This cell installs only missing analysis packages. It leaves the existing PyTorch and torchvision installation unchanged so the notebook does not replace a working CUDA build. LPIPS and pretrained feature extractors are used only for evaluation; the CycleGAN generators and discriminators are trained from random initialization.

In [ ]:
import importlib.util, subprocess, sys
required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
            'PIL': 'pillow', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn',
            'lpips': 'lpips', 'nbconvert': 'nbconvert', 'threadpoolctl': 'threadpoolctl'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
assert importlib.util.find_spec('torch') and importlib.util.find_spec('torchvision'), 'Use the PyTorch template.'


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from contextlib import nullcontext
import os, json, random, time, hashlib, platform, gc, zipfile, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import inception_v3, Inception_V3_Weights, resnet18, ResNet18_Weights
from scipy import linalg
from threadpoolctl import threadpool_limits
from sklearn.metrics import cohen_kappa_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(8, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = device.type == 'cuda'
# Keep the metric extractor in FP32 without TF32 approximation.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print('Python:', platform.python_version(), 'PyTorch:', torch.__version__, 'Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0), 'VRAM GB:', torch.cuda.get_device_properties(0).total_memory/2**30)


### 0.2 Experiment configuration

The main run uses 200 epochs with 512 update pairs per epoch, for 102,400 total updates. An epoch is a fixed update budget rather than one full pass through all 7,038 photos. The shuffled photo schedule continues across epochs so the whole photo folder remains in the sampling pool.

Training resizes images to 286, takes random 256 × 256 crops, and applies random horizontal flips. Inference is deterministic at 256 × 256. The learning rate is 0.0002 for the first 100 epochs and then decays linearly through epoch 200. Batch size 1 follows the standard unpaired CycleGAN setup.

`RUN_TRAINING` is kept as an explicit safety switch. Enable it only after the data and evaluator checks pass. `ALLOW_PARTIAL_EXPORT` should remain false for the reported complete run.

In [ ]:
ROOT = Path.cwd().resolve()  # Edit if your uploaded data is elsewhere.
CONFIG = {
    'run_name': 'vast_256_seed3143_v1', 'seed': 3143,
    'image_size': 256, 'load_size': 286, 'base_channels': 64, 'residual_blocks': 9,
    'epochs': 200, 'steps_per_epoch': 512, 'decay_start': 100, 'batch_size': 1,
    'lr': 2e-4, 'beta1': 0.5, 'beta2': 0.999,
    'lambda_cycle': 10.0, 'lambda_identity': 5.0,
    'gradient_clip': None, 'pool_size': 50, 'amp': True,
    'cache_decoded_images': True, 'workers': 0,
    'checkpoint_minutes': 10, 'evaluate_every': 10, 'selection_images': 300,
    'report_images': 300, 'feature_batch': 16, 'inference_batch': 4,
    'sanity_fid_limit': 1.0, 'max_session_hours': 8.0,
    'gpu_price_per_hour': 0.50,  # Replace with YOUR quoted offer; excludes storage/network/taxes.
    'mifid_epsilon': 0.1, 'mifid_distance_direction': 'real_to_fake',
}
RUN_TRAINING = False           # Enable ONLY after protocol check + benchmark.
RUN_BENCHMARK = True           # Disposable weights; does not become the experiment.
ALLOW_PARTIAL_EXPORT = False   # Explicitly opt in only if you intend to report an unfinished run.
ORGANIZER_METRIC_RULES_CONFIRMED = False  # Must verify MiFID formula + preprocessing with organizer.
TEMPLATE_PATH = ROOT / 'data' / 'sample_submission.csv'
DATA_DIR = ROOT / 'data'
RUN_DIR = ROOT / 'runs' / CONFIG['run_name']
OUT = RUN_DIR / 'outputs'
CKPT = RUN_DIR / 'checkpoints'
assert CONFIG['epochs'] > CONFIG['decay_start'] >= 1
assert CONFIG['image_size'] % 4 == 0 and CONFIG['load_size'] >= CONFIG['image_size']
assert CONFIG['batch_size'] == 1, 'This reproducible sampling plan is designed for batch_size=1.'
assert '/' not in CONFIG['run_name'] and '\\' not in CONFIG['run_name']
for directory in [RUN_DIR, OUT, CKPT]:
    directory.mkdir(parents=True, exist_ok=True)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def json_atomic(path, value):
    temporary = Path(str(path)+'.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

seed_all(CONFIG['seed'])
print(json.dumps(CONFIG, indent=2))
print('Results:', RUN_DIR)


## 1. Validate the two unpaired domains

The notebook checks every image before training. It verifies the expected domain folders, decodes each file as RGB, records the counts, and hashes the dataset identity. A decoding failure stops the run instead of silently reducing the dataset.

The two folders are sampled independently; filenames do not represent paired examples. The recorded run contains 300 Monet images and 7,038 photos.

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png'}
def image_files(folder):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS)

assert (DATA_DIR/'monet_jpg').is_dir() and (DATA_DIR/'photo_jpg').is_dir(), 'Upload both image folders into data/.'
monet_files = image_files(DATA_DIR/'monet_jpg')
photo_files = image_files(DATA_DIR/'photo_jpg')
assert len(monet_files) >= 30 and len(photo_files) >= 30, 'Both domains need at least 30 images.'
assert len({p.stem for p in photo_files}) == len(photo_files), 'Photo stems must be unique for PNG export.'
STAT_PATH = DATA_DIR/'real_stats.npz'
assert STAT_PATH.exists(), 'Upload the supplied real_stats.npz before proceeding.'

fingerprint = hashlib.sha256()
for label, files in [('A', monet_files), ('B', photo_files)]:
    for p in files:
        with PILImage.open(p) as im: im.verify()
        fingerprint.update((label+':'+p.name).encode())
        fingerprint.update(hashlib.sha256(p.read_bytes()).digest())
DATA_HASH = fingerprint.hexdigest()
STATS_HASH = hashlib.sha256(STAT_PATH.read_bytes()).hexdigest()
print('Monet:', len(monet_files), 'Photos:', len(photo_files))
print('Dataset SHA256:', DATA_HASH)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for row, files in enumerate([monet_files, photo_files]):
    for col, p in enumerate(files[:4]):
        with PILImage.open(p) as im: axes[row,col].imshow(im.convert('RGB'))
        axes[row,col].set_title('Monet' if row == 0 else 'Photo'); axes[row,col].axis('off')
plt.tight_layout(); plt.show()


## 2. Validate the evaluator before training

Checkpoint selection is meaningful only when every checkpoint uses the same feature pipeline. This section validates the supplied Monet statistics against torchvision InceptionV3 features using the saved preprocessing protocol: initial resize to 256, tensor resize to 299, ImageNet normalization, and the model's default input transform.

A near-zero FID between the real Monet images and the supplied reference statistics is used as a protocol sanity check. This confirms the local feature implementation, but it does not by itself prove that every external leaderboard detail is identical.

In [ ]:
inception = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=True)
inception.fc = nn.Identity()
inception.eval().to(device).requires_grad_(False)
IM_MEAN = torch.tensor([.485, .456, .406], device=device)[None,:,None,None]
IM_STD = torch.tensor([.229, .224, .225], device=device)[None,:,None,None]
def load_rgb(path, size):
    with PILImage.open(path) as im:
        # Explicit BICUBIC matches PIL's RGB resize default from the previous notebook.
        return np.array(im.convert('RGB').resize((size,size), PILImage.Resampling.BICUBIC), copy=True)

class Files01(Dataset):
    def __init__(self, files, size=256): self.files, self.size = list(files), size
    def __len__(self): return len(self.files)
    def __getitem__(self, index):
        return torch.from_numpy(load_rgb(self.files[index], self.size)).permute(2,0,1).float()/255

@torch.inference_mode()
def features_from_files(files, model=None, target_size=299):
    model = inception if model is None else model
    result = []
    loader = DataLoader(Files01(files), batch_size=CONFIG['feature_batch'], shuffle=False, num_workers=0)
    for x in loader:
        x = x.to(device)
        x = F.interpolate(x, (target_size,target_size), mode='bilinear', align_corners=False)
        y = model((x-IM_MEAN)/IM_STD)
        if hasattr(y, 'logits'): y = y.logits
        result.append(y.float().cpu().numpy())
    result = np.concatenate(result).astype(np.float64)
    assert np.isfinite(result).all(), 'Non-finite metric features.'
    return result

with np.load(STAT_PATH, allow_pickle=False) as stats:
    assert {'mu_real','sigma_real','feats_real'} <= set(stats.files), f'Unexpected stats keys: {stats.files}'
    REF_MU = stats['mu_real'].astype(np.float64)
    REF_COV = stats['sigma_real'].astype(np.float64)
    REF_FEATURES = stats['feats_real'].astype(np.float64)
assert REF_MU.shape == (2048,) and REF_COV.shape == (2048,2048)
assert REF_FEATURES.ndim == 2 and REF_FEATURES.shape[1] == 2048
assert all(np.isfinite(x).all() for x in [REF_MU, REF_COV, REF_FEATURES])


### 2.1 FID computation

For real and generated feature distributions, FID combines the squared distance between their means with a covariance term. The implementation below uses a numerically stable low-rank form of the same FID equation, which avoids repeatedly computing a large 2048 × 2048 matrix square root.

Small negative eigenvalues caused by floating-point rounding are clipped, while substantial invalid values fail validation. The sanity-test result is also compared with SciPy so an implementation error is caught before checkpoint evaluation.

In [ ]:
def make_fid_reference(mu, covariance):
    covariance = (covariance+covariance.T)/2
    with threadpool_limits(limits=8):
        eigenvalues, eigenvectors = linalg.eigh(covariance, check_finite=True)
    assert eigenvalues.min() >= -1e-6*max(1, eigenvalues.max()), 'Reference covariance is not PSD.'
    cutoff = max(1, eigenvalues.max())*1e-12
    keep = eigenvalues > cutoff
    factor = eigenvectors[:,keep]*np.sqrt(np.maximum(eigenvalues[keep],0))[None,:]
    return {'mu': np.asarray(mu), 'factor': factor, 'trace': float(np.trace(covariance)), 'cov': covariance}

def fid_with_reference(reference, fake_features):
    assert len(fake_features) >= 2
    fake_mu = fake_features.mean(0)
    centered = fake_features-fake_mu
    with threadpool_limits(limits=8):
        cross = linalg.svdvals(centered @ reference['factor']).sum()/np.sqrt(len(centered)-1)
    value = np.sum((fake_mu-reference['mu'])**2)+reference['trace']+np.sum(centered**2)/(len(centered)-1)-2*cross
    assert np.isfinite(value) and value >= -0.01, f'Invalid FID: {value}'
    return float(max(0, value))

def scipy_fid(reference, fake_features):
    mu, cov = fake_features.mean(0), np.cov(fake_features, rowvar=False)
    with threadpool_limits(limits=8):
        root = linalg.sqrtm(reference['cov'] @ cov)
    assert np.isfinite(root).all()
    if np.iscomplexobj(root):
        assert np.max(np.abs(root.diagonal().imag)) < .01, 'Large imaginary FID component.'
        root = root.real
    return float(max(0, np.sum((mu-reference['mu'])**2)+np.trace(reference['cov']+cov-2*root)))

FID_REF_A = make_fid_reference(REF_MU, REF_COV)
real_A_features = features_from_files(monet_files)
sanity_fid = fid_with_reference(FID_REF_A, real_A_features)
sanity_scipy = scipy_fid(FID_REF_A, real_A_features)
assert abs(sanity_fid-sanity_scipy) < .05, 'Fast FID disagrees with independent SciPy implementation.'
json_atomic(OUT/'protocol_check.json', {
    'sanity_fid': sanity_fid, 'sanity_scipy': sanity_scipy,
    'stats_sha256': STATS_HASH, 'inception': 'torchvision IMAGENET1K_V1 transform_input=True',
    'initial_size': 256, 'pil_resize': 'BICUBIC', 'tensor_resize': 'bilinear 299 align_corners=False',
    'normalization': 'ImageNet', 'dtype': 'float32 extractor, float64 FID',
})
print('Real-Monet sanity FID:', sanity_fid, 'SciPy:', sanity_scipy)
assert sanity_fid <= CONFIG['sanity_fid_limit'], (
    'STOP: feature protocol does not match real_stats.npz. Do not train or submit yet.')
PROTOCOL_OK = True


### 2.2 Local MiFID convention

The saved local evaluator computes the mean nearest absolute-cosine distance from real features to generated features. With threshold 0.1, the penalty is that mean when it falls below the threshold and 1 otherwise; local MiFID is `FID / penalty`.

This convention is stored explicitly because MiFID implementations can differ in direction, preprocessing, and penalty definition. Checkpoints are selected by fixed-subset FID, so the provisional MiFID rule does not influence model selection. The submitted CSV values and actual leaderboard score are recorded separately from this local diagnostic.

In [ ]:
def mifid_local(fid, real, fake):
    real = real/np.maximum(np.linalg.norm(real, axis=1, keepdims=True), 1e-12)
    fake = fake/np.maximum(np.linalg.norm(fake, axis=1, keepdims=True), 1e-12)
    direction = CONFIG['mifid_distance_direction']
    assert direction in {'real_to_fake','fake_to_real'}
    source, target = (real,fake) if direction == 'real_to_fake' else (fake,real)
    distances = []
    for start in range(0,len(source),128):
        d = np.maximum(0,1-np.abs(source[start:start+128] @ target.T))
        distances.extend(d.min(axis=1).tolist())
    distance = float(np.mean(distances))
    penalty = distance if distance < CONFIG['mifid_epsilon'] else 1.0
    return {'MiFID': float(fid/max(penalty,1e-12)), 'memorization_distance': distance,
            'memorization_penalty': penalty, 'direction': direction, 'epsilon': CONFIG['mifid_epsilon'],
            'rule_confirmed': ORGANIZER_METRIC_RULES_CONFIRMED}

# Preload LPIPS now so a missing dependency/weight download does not spoil a finished training run.
import lpips
lpips_net = lpips.LPIPS(net='alex', verbose=False).eval().requires_grad_(False)  # CPU until final reporting.
print('FID protocol passed. LPIPS ready. MiFID organizer confirmation:', ORGANIZER_METRIC_RULES_CONFIRMED)


## 3. Independent-domain sampling

Monet and photo images are shuffled independently. Photo sampling advances through seeded permutations across epochs, while the smaller Monet domain is reshuffled more frequently. This preserves the unpaired-learning setup and avoids a permanent subset of photos.

Images can be decoded once into memory to reduce repeated JPEG reads. The sampling cursor, random-number states, configuration hash, and data hash are included in resumable checkpoints so an interrupted run can continue consistently.

In [ ]:
class UnpairedDataset(Dataset):
    def __init__(self, a_files, b_files, cfg):
        self.files = [list(a_files), list(b_files)]
        self.cfg = cfg
        self.cache = [None,None]
        if cfg['cache_decoded_images']:
            self.cache = [[load_rgb(p,cfg['load_size']) for p in files] for files in self.files]
        self.augment = transforms.Compose([
            transforms.RandomCrop(cfg['image_size']), transforms.RandomHorizontalFlip(),
            transforms.ToTensor(), transforms.Normalize((.5,)*3,(.5,)*3)])
        self.set_epoch(0)

    def schedule(self, length, domain_seed, start, count):
        indices, cursor = [], start
        while len(indices) < count:
            cycle, offset = divmod(cursor,length)
            order = np.random.default_rng(domain_seed+cycle).permutation(length)
            take = min(length-offset,count-len(indices))
            indices.extend(order[offset:offset+take].tolist()); cursor += take
        return indices

    def set_epoch(self, epoch):
        n = self.cfg['steps_per_epoch']
        self.indices = [self.schedule(len(f),self.cfg['seed']+d*100000,epoch*n,n)
                        for d,f in enumerate(self.files)]
    def __len__(self): return self.cfg['steps_per_epoch']
    def __getitem__(self,index):
        samples = []
        for domain in (0,1):
            j = self.indices[domain][index]
            rgb = self.cache[domain][j] if self.cache[domain] is not None else load_rgb(self.files[domain][j],self.cfg['load_size'])
            samples.append(self.augment(PILImage.fromarray(rgb)))
        return samples[0],samples[1]

train_dataset = UnpairedDataset(monet_files, photo_files, CONFIG)
print('Cached data ready. Optimization steps:', CONFIG['epochs']*CONFIG['steps_per_epoch'])
print('Full-photo traversals planned:', CONFIG['epochs']*CONFIG['steps_per_epoch']/len(photo_files))


## 4. CycleGAN architecture

The model contains two generators and two discriminators. Each generator has two downsampling stages, nine residual blocks at the 64 × 64 feature resolution, and two upsampling stages. Instance normalization supports batch size 1, reflection padding reduces boundary artifacts, and `tanh` maps output pixels to `[-1, 1]`.

Each PatchGAN discriminator returns a grid of realism scores rather than one score for the whole image. The architecture follows the CycleGAN reference design, while the training schedule, image pool, evaluation protocol, and run configuration are recorded explicitly for this experiment.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.block = nn.Sequential(nn.ReflectionPad2d(1),nn.Conv2d(c,c,3),nn.InstanceNorm2d(c),nn.ReLU(True),
                                   nn.ReflectionPad2d(1),nn.Conv2d(c,c,3),nn.InstanceNorm2d(c))
    def forward(self,x): return x+self.block(x)

class Generator(nn.Module):
    def __init__(self,channels=64,blocks=9):
        super().__init__()
        layers = [nn.ReflectionPad2d(3),nn.Conv2d(3,channels,7),nn.InstanceNorm2d(channels),nn.ReLU(True)]
        c = channels
        for _ in range(2):
            layers += [nn.Conv2d(c,c*2,3,2,1),nn.InstanceNorm2d(c*2),nn.ReLU(True)]; c *= 2
        layers += [ResidualBlock(c) for _ in range(blocks)]
        for _ in range(2):
            layers += [nn.ConvTranspose2d(c,c//2,3,2,1,output_padding=1),nn.InstanceNorm2d(c//2),nn.ReLU(True)]; c //= 2
        layers += [nn.ReflectionPad2d(3),nn.Conv2d(c,3,7),nn.Tanh()]
        self.net = nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

class Discriminator(nn.Module):
    def __init__(self,channels=64):
        super().__init__()
        layers = [nn.Conv2d(3,channels,4,2,1),nn.LeakyReLU(.2,True)]
        for multiplier,stride in [(2,2),(4,2),(8,1)]:
            previous = channels*(multiplier//2)
            layers += [nn.Conv2d(previous,channels*multiplier,4,stride,1),
                       nn.InstanceNorm2d(channels*multiplier),nn.LeakyReLU(.2,True)]
        layers += [nn.Conv2d(channels*8,1,4,1,1)]
        self.net = nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def initialize_weights(module):
    if isinstance(module,(nn.Conv2d,nn.ConvTranspose2d)):
        nn.init.normal_(module.weight,0,.02)
        if module.bias is not None: nn.init.zeros_(module.bias)

def make_networks(cfg):
    networks = {'G_A2B': Generator(cfg['base_channels'],cfg['residual_blocks']),
                'G_B2A': Generator(cfg['base_channels'],cfg['residual_blocks']),
                'D_A': Discriminator(cfg['base_channels']), 'D_B': Discriminator(cfg['base_channels'])}
    for model in networks.values(): model.apply(initialize_weights); model.to(device)
    return networks


## 5. Losses, schedule, and replay buffers

The generators minimize least-squares adversarial loss in both directions. Cycle-consistency loss encourages an image translated to the other domain and back to reconstruct its source. Identity loss discourages unnecessary changes to an image that is already in the generator's target domain.

The total generator objective uses cycle weight 10 and identity weight 5. Both discriminators train on current real images and a replay pool of 50 previously generated images. Discriminator gradients are disabled during generator updates. AMP is enabled, and any non-finite loss or gradient stops the run instead of being hidden.

Although the direct loss functions are simple, GAN training is not a convex optimization problem. Loss curves therefore need to be interpreted together with FID, visual examples, and stability measurements.

In [ ]:
class ImagePool:
    def __init__(self,capacity): self.capacity,self.images = capacity,[]
    def query(self,batch):
        if self.capacity == 0: return batch.detach()
        result = []
        for image in batch.detach():
            image = image.unsqueeze(0)
            if len(self.images) < self.capacity:
                self.images.append(image.clone()); result.append(image)
            elif random.random() > .5:
                index = random.randrange(self.capacity)
                result.append(self.images[index].clone()); self.images[index] = image.clone()
            else: result.append(image)
        return torch.cat(result)

def lr_for_epoch(epoch,cfg):
    if epoch < cfg['decay_start']: return cfg['lr']
    return cfg['lr']*(cfg['epochs']-epoch)/(cfg['epochs']-cfg['decay_start'])

def make_training_state(cfg):
    nets = make_networks(cfg)
    gp = list(nets['G_A2B'].parameters())+list(nets['G_B2A'].parameters())
    dp = list(nets['D_A'].parameters())+list(nets['D_B'].parameters())
    og = torch.optim.Adam(gp,lr=cfg['lr'],betas=(cfg['beta1'],cfg['beta2']))
    od = torch.optim.Adam(dp,lr=cfg['lr'],betas=(cfg['beta1'],cfg['beta2']))
    amp = cfg['amp'] and device.type == 'cuda'
    dtype = torch.bfloat16 if amp and torch.cuda.is_bf16_supported() else torch.float16
    return {'nets': nets,'gp': gp,'dp': dp,'og': og,'od': od,'amp': amp,'dtype': dtype,
            'sg': torch.amp.GradScaler('cuda',enabled=amp and dtype == torch.float16),
            'sd': torch.amp.GradScaler('cuda',enabled=amp and dtype == torch.float16),
            'pa': ImagePool(cfg['pool_size']), 'pb': ImagePool(cfg['pool_size'])}

def amp_context(state):
    return torch.autocast('cuda',dtype=state['dtype']) if state['amp'] else nullcontext()

def update_optimizer(loss,optimizer,scaler,parameters,clip=None):
    if not torch.isfinite(loss): raise FloatingPointError('Non-finite loss.')
    scaler.scale(loss).backward(); scaler.unscale_(optimizer)
    norm = nn.utils.clip_grad_norm_(parameters,float('inf') if clip is None else clip,error_if_nonfinite=True)
    scaler.step(optimizer); scaler.update()
    return float(norm.detach())


In [ ]:
def train_step(state,real_A,real_B,cfg):
    n = state['nets']; ga,gb,da,db = [n[k] for k in ['G_A2B','G_B2A','D_A','D_B']]
    real_A,real_B = real_A.to(device),real_B.to(device)
    for p in state['dp']: p.requires_grad_(False)
    state['og'].zero_grad(set_to_none=True)
    with amp_context(state):
        fake_B,fake_A = ga(real_A),gb(real_B)
        cycle_A,cycle_B = gb(fake_B),ga(fake_A)
        pred_A,pred_B = da(fake_A),db(fake_B)
        adv = F.mse_loss(pred_A.float(),torch.ones_like(pred_A,dtype=torch.float32))
        adv += F.mse_loss(pred_B.float(),torch.ones_like(pred_B,dtype=torch.float32))
        cycle = F.l1_loss(cycle_A.float(),real_A)+F.l1_loss(cycle_B.float(),real_B)
        identity = F.l1_loss(gb(real_A).float(),real_A)+F.l1_loss(ga(real_B).float(),real_B)
        loss_G = adv+cfg['lambda_cycle']*cycle+cfg['lambda_identity']*identity
    grad_G = update_optimizer(loss_G,state['og'],state['sg'],state['gp'],cfg['gradient_clip'])

    for p in state['dp']: p.requires_grad_(True)
    state['od'].zero_grad(set_to_none=True)
    with amp_context(state):
        pred_real_A,pred_real_B = da(real_A),db(real_B)
        pred_fake_A,pred_fake_B = da(state['pa'].query(fake_A)),db(state['pb'].query(fake_B))
        d_A = .5*(F.mse_loss(pred_real_A.float(),torch.ones_like(pred_real_A,dtype=torch.float32))+
                  F.mse_loss(pred_fake_A.float(),torch.zeros_like(pred_fake_A,dtype=torch.float32)))
        d_B = .5*(F.mse_loss(pred_real_B.float(),torch.ones_like(pred_real_B,dtype=torch.float32))+
                  F.mse_loss(pred_fake_B.float(),torch.zeros_like(pred_fake_B,dtype=torch.float32)))
        loss_D = d_A+d_B
    grad_D = update_optimizer(loss_D,state['od'],state['sd'],state['dp'],cfg['gradient_clip'])
    return {'G_loss':float(loss_G.detach()),'D_A_loss':float(d_A.detach()),'D_B_loss':float(d_B.detach()),
            'adversarial':float(adv.detach()),'cycle_L1':float(cycle.detach()),'identity_L1':float(identity.detach()),
            'G_grad':grad_G,'D_grad':grad_D}


## 6. Short benchmark

Before the full run, this section performs a small number of warm-up and measured updates with disposable model weights. It estimates training throughput and memory use without changing the experiment checkpoint.

The benchmark covers training updates only. Evaluation, plotting, checkpoint writing, and final inference add extra time, so the estimate is a planning aid rather than a guaranteed total runtime.

In [ ]:
if RUN_BENCHMARK:
    assert device.type == 'cuda', 'Benchmark and training require a CUDA-capable GPU.'
    seed_all(CONFIG['seed'])
    bench = make_training_state(CONFIG)
    x,y = train_dataset[0]
    print('Generator parameters:',sum(p.numel() for key in ['G_A2B','G_B2A'] for p in bench['nets'][key].parameters()))
    print('Discriminator parameters:',sum(p.numel() for key in ['D_A','D_B'] for p in bench['nets'][key].parameters()))
    print('Autocast:',bench['amp'],str(bench['dtype']))
    for _ in range(5): train_step(bench,x.unsqueeze(0),y.unsqueeze(0),CONFIG)
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    for i in range(30):
        x,y = train_dataset[i % len(train_dataset)]
        train_step(bench,x.unsqueeze(0),y.unsqueeze(0),CONFIG)
    torch.cuda.synchronize()
    seconds_per_step = (time.perf_counter()-t0)/30
    estimated_hours = seconds_per_step*CONFIG['steps_per_epoch']*CONFIG['epochs']/3600
    estimate = {'seconds_per_step':seconds_per_step,'training_only_hours':estimated_hours,
                'training_only_gpu_cost':estimated_hours*CONFIG['gpu_price_per_hour'],
                'peak_memory_mb':torch.cuda.max_memory_allocated()/2**20,
                'gpu':torch.cuda.get_device_name(0)}
    json_atomic(OUT/'benchmark.json',estimate)
    print(json.dumps(estimate,indent=2))
    del bench,x,y; gc.collect(); torch.cuda.empty_cache(); seed_all(CONFIG['seed'])
else:
    print('Benchmark skipped. Review an existing benchmark.json before enabling training.')


## 7. Checkpoints and evidence logs

`latest.pt` is the resumable training checkpoint. It stores all four networks, optimizer and scaler states, replay buffers, random-number states, the sampling cursor, partial-epoch summaries, hashes, and timing data.

The inference checkpoint is selected by the lowest FID on the same fixed 300-photo subset at every evaluation. The fixed subset supports a fair checkpoint comparison; it is not a cap on the images used for training. Training history and checkpoint-selection records are appended as run evidence.

In [ ]:
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()
def log_event(event,**fields):
    record = {'time_utc':datetime.now(timezone.utc).isoformat(),'event':event,**fields}
    with (RUN_DIR/'raw_events.jsonl').open('a',encoding='utf-8') as stream:
        stream.write(json.dumps(record,allow_nan=False)+'\n'); stream.flush()
    print(event,fields,flush=True)

def torch_atomic(value,path):
    temporary = Path(str(path)+'.tmp'); torch.save(value,temporary); temporary.replace(path)

def rng_snapshot():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}

def restore_rng(rng):
    random.setstate(rng['python']); np.random.set_state(rng['numpy']); torch.set_rng_state(rng['torch'])
    if torch.cuda.is_available() and rng['cuda']: torch.cuda.set_rng_state_all(rng['cuda'])

def save_latest(state,progress):
    payload = {'config':CONFIG,'config_hash':CONFIG_HASH,'data_hash':DATA_HASH,'stats_hash':STATS_HASH,
               'models':{k:v.state_dict() for k,v in state['nets'].items()},
               'optimizers':{'G':state['og'].state_dict(),'D':state['od'].state_dict()},
               'scalers':{'G':state['sg'].state_dict(),'D':state['sd'].state_dict()},
               'pools':{'A':[x.cpu() for x in state['pa'].images],'B':[x.cpu() for x in state['pb'].images]},
               'rng':rng_snapshot(),'progress':copy.deepcopy(progress)}
    torch_atomic(payload,CKPT/'latest.pt')

def load_latest(state):
    # Load only checkpoints produced by YOU. weights_only=False permits stored RNG/optimizer objects.
    payload = torch.load(CKPT/'latest.pt',map_location='cpu',weights_only=False)
    assert payload['config_hash'] == CONFIG_HASH, 'Config changed. Use a NEW run_name for a new experiment.'
    assert payload['data_hash'] == DATA_HASH and payload['stats_hash'] == STATS_HASH, 'Data/reference changed.'
    for key,model in state['nets'].items(): model.load_state_dict(payload['models'][key])
    state['og'].load_state_dict(payload['optimizers']['G']); state['od'].load_state_dict(payload['optimizers']['D'])
    state['sg'].load_state_dict(payload['scalers']['G']); state['sd'].load_state_dict(payload['scalers']['D'])
    state['pa'].images = [x.to(device) for x in payload['pools']['A']]
    state['pb'].images = [x.to(device) for x in payload['pools']['B']]
    restore_rng(payload['rng'])
    return payload['progress']

def fresh_progress():
    return {'epoch':0,'step':0,'sums':{},'max_grad':0.0,'history':[],'selection':[],
            'best_fid':None,'training_seconds':0.0,'total_seconds':0.0,'completed_steps':0,
            'nan_inf_count':0,'peak_memory_mb':0.0,'pending_evaluation':None}


In [ ]:
def tensor_pixels(x):
    return ((x.detach().float().cpu().clamp(-1,1)+1)*127.5).round().to(torch.uint8).permute(1,2,0).numpy()

def save_png(tensor,path):
    PILImage.fromarray(tensor_pixels(tensor)).save(path)

@torch.inference_mode()
def generate_files(generator,inputs,destination):
    destination.mkdir(parents=True,exist_ok=True)
    previous_mode = generator.training; generator.eval()
    outputs = []
    loader = DataLoader(Files01(inputs,CONFIG['image_size']),batch_size=CONFIG['inference_batch'],shuffle=False,num_workers=0)
    cursor = 0
    for x in loader:
        # FP32 inference; no pretrained generator, blend, edits, or manual selection.
        translated = generator(x.to(device)*2-1)
        for image in translated:
            p = destination/(inputs[cursor].stem+'.png')
            save_png(image,p); outputs.append(p); cursor += 1
    generator.train(previous_mode)
    assert len(outputs) == len(inputs)
    return outputs

selection_rng = np.random.default_rng(CONFIG['seed']+700)
selection_indices = sorted(selection_rng.choice(len(photo_files),min(CONFIG['selection_images'],len(photo_files)),replace=False))
selection_inputs = [photo_files[i] for i in selection_indices]
json_atomic(OUT/'selection_manifest.json',{'filenames':[p.name for p in selection_inputs],'data_hash':DATA_HASH})

def evaluate_checkpoint(state,progress,completed_epoch):
    files = generate_files(state['nets']['G_B2A'],selection_inputs,OUT/'selection_preview')
    feats = features_from_files(files)
    fid = fid_with_reference(FID_REF_A,feats)
    score = {'epoch':completed_epoch,'FID':fid,'images':len(files)}
    progress['selection'].append(score)
    pd.DataFrame(progress['selection']).to_csv(OUT/'checkpoint_selection.csv',index=False)
    log_event('checkpoint_selection',**score)
    if progress['best_fid'] is None or fid < progress['best_fid']:
        progress['best_fid'] = fid
        torch_atomic({'config':CONFIG,'data_hash':DATA_HASH,'stats_hash':STATS_HASH,
                      'selected_epoch':completed_epoch,'selection_FID':fid,
                      'models':{k:v.state_dict() for k,v in state['nets'].items()}},CKPT/'best_fid.pt')
        log_event('new_best',**score)


### 7.1 Training and resume behavior

With `RUN_TRAINING=False`, the notebook completes preflight checks without starting the long run. Setting it to true starts training or resumes from `latest.pt` when a compatible checkpoint is present.

The saved configuration, data hash, epoch, step cursor, and random states are checked before resume. This protects the result from accidentally continuing with different data or hyperparameters. Final export remains blocked until all planned epochs finish unless partial export is explicitly allowed.

In [ ]:
def run_training():
    assert PROTOCOL_OK and device.type == 'cuda'
    if CONFIG['workers'] != 0:
        raise ValueError('Keep workers=0 for reproducible mid-epoch notebook resumes.')
    assert not ((CKPT/'best_fid.pt').exists() and not (CKPT/'latest.pt').exists()), 'Missing latest resume checkpoint.'
    lock = RUN_DIR/'training.lock'
    try: lock_fd = os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
    except FileExistsError:
        raise RuntimeError('training.lock exists. Verify no other training process is running before removing this stale lock.')
    os.write(lock_fd,str(os.getpid()).encode()); os.close(lock_fd)
    state,progress = None,None
    try:
        seed_all(CONFIG['seed']); state = make_training_state(CONFIG)
        progress = load_latest(state) if (CKPT/'latest.pt').exists() else fresh_progress()
        json_atomic(RUN_DIR/'config.json',CONFIG)
        if not (RUN_DIR/'environment.txt').exists():
            packages = subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
            hardware = subprocess.check_output(['nvidia-smi'],text=True)
            (RUN_DIR/'environment.txt').write_text(packages+'\n'+hardware,encoding='utf-8')
        if not (RUN_DIR/'manifest.json').exists():
            json_atomic(RUN_DIR/'manifest.json',{'data_hash':DATA_HASH,'stats_hash':STATS_HASH,'config_hash':CONFIG_HASH,
                'GPU':torch.cuda.get_device_name(0),'python':platform.python_version(),'torch':torch.__version__,
                'A_count':len(monet_files),'B_count':len(photo_files),
                'generator_parameters':sum(p.numel() for p in state['gp']),
                'discriminator_parameters':sum(p.numel() for p in state['dp'])})
        log_event('session_started',epoch=progress['epoch'],step=progress['step'])
        torch.cuda.reset_peak_memory_stats()
        session_start = time.perf_counter(); last_save = session_start
        last_accounted = session_start
        def account_wall_time():
            nonlocal last_accounted
            now = time.perf_counter()
            progress['total_seconds'] += now-last_accounted
            last_accounted = now
        if progress.get('pending_evaluation') is not None:
            rng = rng_snapshot()
            evaluate_checkpoint(state,progress,progress['pending_evaluation'])
            restore_rng(rng)
            progress['pending_evaluation'] = None
            account_wall_time()
            save_latest(state,progress)
        for epoch in range(progress['epoch'],CONFIG['epochs']):
            train_dataset.set_epoch(epoch)
            start_step = progress['step'] if progress['epoch'] == epoch else 0
            lr = lr_for_epoch(epoch,CONFIG)
            for optimizer in [state['og'],state['od']]:
                for group in optimizer.param_groups: group['lr'] = lr
            for model in state['nets'].values(): model.train()
            # Supply a separate RNG so constructing a resume loader does not consume model RNG.
            loader = DataLoader(train_dataset,batch_size=1,shuffle=False,num_workers=0,
                                sampler=range(start_step,len(train_dataset)),
                                generator=torch.Generator().manual_seed(CONFIG['seed']+epoch),pin_memory=True)
            for step,(a,b) in enumerate(loader,start=start_step):
                t = time.perf_counter()
                values = train_step(state,a,b,CONFIG)
                # .item() inside train_step synchronizes device work for this timing.
                progress['training_seconds'] += time.perf_counter()-t
                for key,value in values.items(): progress['sums'][key] = progress['sums'].get(key,0)+value
                progress['max_grad'] = max(progress['max_grad'],values['G_grad'],values['D_grad'])
                progress['completed_steps'] += 1
                progress['epoch'],progress['step'] = epoch,step+1
                progress['peak_memory_mb'] = max(progress['peak_memory_mb'],torch.cuda.max_memory_allocated()/2**20)
                now = time.perf_counter()
                if now-last_save >= CONFIG['checkpoint_minutes']*60:
                    account_wall_time()
                    save_latest(state,progress); last_save = now
                if now-session_start >= CONFIG['max_session_hours']*3600:
                    account_wall_time()
                    save_latest(state,progress)
                    log_event('session_paused',epoch=epoch+1,step=step+1,reason='time_budget')
                    return progress
            row = {'epoch':epoch+1,'lr':lr,**{k:v/CONFIG['steps_per_epoch'] for k,v in progress['sums'].items()},
                   'gradient_norm_max':progress['max_grad']}
            progress['history'].append(row); log_event('epoch_finished',**row)
            progress['epoch'],progress['step'],progress['sums'],progress['max_grad'] = epoch+1,0,{},0.0
            pd.DataFrame(progress['history']).to_csv(OUT/'training_history.csv',index=False)
            due = (epoch+1) % CONFIG['evaluate_every'] == 0 or epoch+1 == CONFIG['epochs']
            progress['pending_evaluation'] = epoch+1 if due else None
            account_wall_time()
            save_latest(state,progress)  # Save before evaluation, which may fail independently.
            if due:
                rng = rng_snapshot()
                evaluate_checkpoint(state,progress,epoch+1)
                restore_rng(rng)
                progress['pending_evaluation'] = None
                account_wall_time()
                save_latest(state,progress)
            last_save = time.perf_counter()
        account_wall_time()
        save_latest(state,progress); log_event('training_complete',steps=progress['completed_steps'])
        return progress
    except (KeyboardInterrupt,Exception) as error:
        if state is not None and progress is not None:
            if isinstance(error,FloatingPointError) or 'non-finite' in str(error).lower():
                progress['nan_inf_count'] += 1
                log_event('nonfinite_failure',message=str(error))
                # Do not overwrite a good checkpoint with potentially corrupted parameters.
            # Never snapshot inside an interrupted/failed optimizer step. Resume the last GOOD save.
            log_event('session_failed',type=type(error).__name__,message=str(error))
        raise
    finally:
        lock.unlink(missing_ok=True)
        if state is not None: del state
        gc.collect(); torch.cuda.empty_cache()

if RUN_TRAINING:
    progress = run_training()
else:
    print('Preflight complete. Training is OFF. Review benchmark.json, then enable RUN_TRAINING and save.')


## 8. Training diagnostics

The plots combine generator and discriminator losses, cycle and identity terms, gradient norms, learning rate, and checkpoint FID. GAN losses are expected to fluctuate, so a smooth loss curve is not treated as proof of image quality.

For the recorded run, fixed-subset FID improved from 143.945067 at epoch 10 to 103.877408 at epoch 190, then increased slightly to 104.728361 at epoch 200. This is why epoch 190 is selected instead of the final epoch.

In [ ]:
if (OUT/'training_history.csv').exists():
    history = pd.read_csv(OUT/'training_history.csv')
    fig,axs = plt.subplots(2,2,figsize=(12,8))
    for k in ['G_loss','D_A_loss','D_B_loss']: axs[0,0].plot(history.epoch,history[k],label=k)
    for k in ['cycle_L1','identity_L1']: axs[0,1].plot(history.epoch,history[k],label=k)
    for k in ['G_grad','D_grad']: axs[1,0].plot(history.epoch,history[k],label=k)
    axs[1,1].plot(history.epoch,history.lr,label='learning rate')
    for ax in axs.flat: ax.legend(); ax.set_xlabel('Optimization epoch')
    plt.tight_layout(); plt.savefig(OUT/'loss_gradient_lr_curves.png',dpi=150); plt.show()
    if (OUT/'checkpoint_selection.csv').exists():
        selection = pd.read_csv(OUT/'checkpoint_selection.csv')
        selection.plot(x='epoch',y='FID',title='Fixed-subset checkpoint FID')
        plt.savefig(OUT/'checkpoint_fid_curve.png',dpi=150,bbox_inches='tight'); plt.show()


## 9. Select the best checkpoint and export every photo

The selected generator comes from the checkpoint with the lowest fixed-subset FID. Every source photo is then translated in deterministic filename order with that same generator. Outputs are written as PNG files without blending, enhancement, or manual filtering.

The recorded epoch-190 export contains all 7,038 photo-to-Monet predictions. The inference manifest links each source filename to its generated filename and records the checkpoint, selected epoch, configuration, and data identity.

In [ ]:
FINAL_READY = False
selected = None
if (CKPT/'best_fid.pt').exists() and (CKPT/'latest.pt').exists():
    latest = torch.load(CKPT/'latest.pt',map_location='cpu',weights_only=False)
    progress = latest['progress']
    assert latest['config_hash'] == CONFIG_HASH and latest['data_hash'] == DATA_HASH and latest['stats_hash'] == STATS_HASH
    completed = progress['epoch'] >= CONFIG['epochs'] and progress['step'] == 0 and progress.get('pending_evaluation') is None
    if completed or ALLOW_PARTIAL_EXPORT:
        selected = torch.load(CKPT/'best_fid.pt',map_location='cpu',weights_only=False)
        assert selected['data_hash'] == DATA_HASH and selected['stats_hash'] == STATS_HASH
        ga = Generator(CONFIG['base_channels'],CONFIG['residual_blocks']).to(device)
        gb = Generator(CONFIG['base_channels'],CONFIG['residual_blocks']).to(device)
        ga.load_state_dict(selected['models']['G_A2B']); gb.load_state_dict(selected['models']['G_B2A'])
        ga.eval().requires_grad_(False); gb.eval().requires_grad_(False)
        FINAL = OUT/f"selected_epoch_{selected['selected_epoch']:03d}"
        FINAL.mkdir(exist_ok=True)
        all_monet_outputs = generate_files(gb,photo_files,FINAL/'photo_to_monet_all')
        assert len(all_monet_outputs) == len(photo_files)
        json_atomic(FINAL/'inference_manifest.json',{
            'checkpoint':'checkpoints/best_fid.pt','selected_epoch':selected['selected_epoch'],
            'selection_FID':selected['selection_FID'],'complete_training':completed,
            'data_hash':DATA_HASH,'stats_hash':STATS_HASH,
            'input_output':[{'source':p.name,'output':q.name} for p,q in zip(photo_files,all_monet_outputs)]})
        del latest; gc.collect()
        FINAL_READY = True
        print('Generated all photos:',len(all_monet_outputs),'Selected epoch:',selected['selected_epoch'])
    else:
        print('Training paused/incomplete. Resume first, or explicitly enable ALLOW_PARTIAL_EXPORT.')
else:
    print('No trained checkpoint yet; final export skipped.')


In [ ]:
if FINAL_READY:
    all_fake_A_features = features_from_files(all_monet_outputs)
    final_FID = fid_with_reference(FID_REF_A,all_fake_A_features)
    mifid_result = mifid_local(final_FID,REF_FEATURES,all_fake_A_features)
    kaggle_measurements = {'FID':final_FID,**mifid_result,'generated_images':len(all_monet_outputs),
                           'selected_epoch':selected['selected_epoch'],'protocol_sanity_FID':sanity_fid,
                           'status':'LOCAL_MEASUREMENTS_NOT_KAGGLE_SCORE'}
    json_atomic(FINAL/'kaggle_metrics_local.json',kaggle_measurements)
    pd.DataFrame([{'FID':final_FID,'MiFID':mifid_result['MiFID']}]).to_csv(FINAL/'metrics_candidate_NOT_FOR_UPLOAD.csv',index=False)
    with zipfile.ZipFile(FINAL/'generated_photo_to_monet.zip','w',zipfile.ZIP_STORED) as archive:
        for p in all_monet_outputs: archive.write(p,arcname=p.name)
    print(json.dumps(kaggle_measurements,indent=2))


## 10. Metrics in both directions

This section evaluates fixed, seeded subsets of up to 300 images per domain so both translation directions can be compared with the same sample count. It reports directional FID, KID, generative precision and recall, cycle L1, LPIPS change, and content-feature cosine similarity.

These metrics describe different properties. FID and KID compare feature distributions; precision and recall summarize realism and coverage; cycle L1 checks reconstruction; LPIPS measures perceptual change from the input; and cosine similarity estimates content retention. None of them is paired-image accuracy because no paired target image exists.

In [ ]:
def fixed_subset(files,count,seed):
    indices = sorted(np.random.default_rng(seed).choice(len(files),min(count,len(files)),replace=False))
    return [files[i] for i in indices]

def lowrank_fid(real,fake):
    rc,fc = real-real.mean(0),fake-fake.mean(0)
    with threadpool_limits(limits=8):
        cross = linalg.svdvals(rc @ fc.T).sum()/np.sqrt((len(real)-1)*(len(fake)-1))
    value = np.sum((real.mean(0)-fake.mean(0))**2)+np.sum(rc**2)/(len(real)-1)+np.sum(fc**2)/(len(fake)-1)-2*cross
    return float(max(0,value))

def kid(real,fake,subsets=20,subset_size=100,seed=3143):
    rng = np.random.default_rng(seed); n = min(subset_size,len(real),len(fake)); values = []
    assert n >= 2
    for _ in range(subsets):
        x,y = real[rng.choice(len(real),n,False)],fake[rng.choice(len(fake),n,False)]
        xx,yy,xy = (x@x.T/x.shape[1]+1)**3,(y@y.T/y.shape[1]+1)**3,(x@y.T/x.shape[1]+1)**3
        values.append((xx.sum()-np.trace(xx))/(n*(n-1))+(yy.sum()-np.trace(yy))/(n*(n-1))-2*xy.mean())
    return float(np.mean(values)),float(np.std(values,ddof=1))

def distances(x,y):
    return np.maximum(0,np.sum(x*x,1)[:,None]+np.sum(y*y,1)[None,:]-2*x@y.T)

def precision_recall(real,fake,k=3):
    assert min(len(real),len(fake)) > k
    rr,ff,rf = distances(real,real),distances(fake,fake),distances(real,fake)
    # Remove self-distance explicitly, including when duplicate features exist.
    np.fill_diagonal(rr,np.inf); np.fill_diagonal(ff,np.inf)
    r = np.partition(rr,k-1,axis=1)[:,k-1]; f = np.partition(ff,k-1,axis=1)[:,k-1]
    return float((rf <= r[:,None]).any(0).mean()),float((rf <= f[None,:]).any(1).mean())

def content_cosine(x,y):
    return float(np.mean(np.sum(x*y,1)/np.maximum(np.linalg.norm(x,axis=1)*np.linalg.norm(y,axis=1),1e-12)))

@torch.inference_mode()
def paired_perceptual_and_cycle(inputs,outputs,forward,reverse):
    lpips_net.to(device)
    values,cycles = [],[]
    for start in range(0,len(inputs),CONFIG['inference_batch']):
        inp,oup = inputs[start:start+CONFIG['inference_batch']],outputs[start:start+CONFIG['inference_batch']]
        x = torch.stack([Files01([p])[0] for p in inp]).to(device)*2-1
        y = torch.stack([Files01([p])[0] for p in oup]).to(device)*2-1
        values.extend(lpips_net(x,y).reshape(-1).cpu().tolist())
        cycle = reverse(forward(x))
        cycles.extend((cycle-x).abs().mean((1,2,3)).cpu().tolist())
    lpips_net.cpu()
    return float(np.mean(values)),float(np.mean(cycles))


In [ ]:
if FINAL_READY:
    report_A = fixed_subset(monet_files,CONFIG['report_images'],CONFIG['seed']+800)
    report_B = fixed_subset(photo_files,CONFIG['report_images'],CONFIG['seed']+900)
    report_fake_B = generate_files(ga,report_A,FINAL/'pred_A2B')
    all_by_source = {p.name:q for p,q in zip(photo_files,all_monet_outputs)}
    report_fake_A = [all_by_source[p.name] for p in report_B]
    # Keep direct B2A evaluation copies alongside A2B results, without recompression.
    (FINAL/'pred_B2A').mkdir(exist_ok=True)
    import shutil
    for p in report_fake_A: shutil.copy2(p,FINAL/'pred_B2A'/p.name)
    report_fake_A = [FINAL/'pred_B2A'/p.name for p in report_fake_A]
    domain_files = {'A':report_A,'B':report_B,'fake_A':report_fake_A,'fake_B':report_fake_B}
    inc_features = {k:features_from_files(files) for k,files in domain_files.items()}
    resnet = resnet18(weights=ResNet18_Weights.DEFAULT).eval().to(device).requires_grad_(False)
    resnet.fc = nn.Identity()
    res_features = {k:features_from_files(files,resnet,target_size=224) for k,files in domain_files.items()}
    report_rows = []
    for direction,real_key,fake_key,input_key,forward,reverse in [
        ('A2B_Monet_to_Photo','B','fake_B','A',ga,gb),('B2A_Photo_to_Monet','A','fake_A','B',gb,ga)]:
        kid_mean,kid_std = kid(inc_features[real_key],inc_features[fake_key])
        precision,recall = precision_recall(res_features[real_key],res_features[fake_key])
        perceptual,cycle_l1 = paired_perceptual_and_cycle(domain_files[input_key],domain_files[fake_key],forward,reverse)
        report_rows.append({'direction':direction,'real_count':len(domain_files[real_key]),'fake_count':len(domain_files[fake_key]),
            'FID_subset':lowrank_fid(inc_features[real_key],inc_features[fake_key]),'KID_mean':kid_mean,'KID_std':kid_std,
            'generative_precision':precision,'generative_recall':recall,'cycle_L1':cycle_l1,
            'LPIPS_input_vs_translation':perceptual,'content_cosine':content_cosine(res_features[input_key],res_features[fake_key])})
    directional = pd.DataFrame(report_rows)
    directional.to_csv(FINAL/'directional_metrics.csv',index=False)
    display(directional)
    del resnet; gc.collect(); torch.cuda.empty_cache()


### 10.1 Translation and cycle-reconstruction examples

The grid uses fixed samples rather than hand-picked successes. Each column shows a source image, its direct translation, and the cycle reconstruction.

The useful questions are whether the translation changes domain style, whether important scene layout survives, whether repeated textures or color casts appear, and whether the reconstruction returns to the original content. A strong cycle reconstruction alone does not guarantee a realistic target-domain translation.

In [ ]:
if FINAL_READY:
    fig,axes = plt.subplots(6,4,figsize=(12,16))
    for domain,inputs,outputs,forward,reverse,row0 in [
        ('Monet',report_A,report_fake_B,ga,gb,0),('Photo',report_B,report_fake_A,gb,ga,3)]:
        for j in range(4):
            x = Files01([inputs[j]])[0].unsqueeze(0).to(device)*2-1
            with torch.inference_mode(): reconstruction = reverse(forward(x))[0]
            axes[row0,j].imshow(tensor_pixels(x[0])); axes[row0,j].set_title('Source '+domain)
            with PILImage.open(outputs[j]) as im: axes[row0+1,j].imshow(im)
            axes[row0+1,j].set_title('Direct translation')
            axes[row0+2,j].imshow(tensor_pixels(reconstruction)); axes[row0+2,j].set_title('Cycle reconstruction')
            for r in range(row0,row0+3): axes[r,j].axis('off')
    plt.tight_layout(); plt.savefig(FINAL/'translation_cycle_grid.png',dpi=140); plt.show()


## 11. Blinded human audit

The notebook prepares 30 fixed source/translation pairs: 15 Monet-to-photo and 15 photo-to-Monet. Their order and direction labels are hidden from two independent raters. Each rater scores style, content preservation, and freedom from artifacts from 1 to 5 without seeing the other person's ratings.

After both raters finish, the report should include mean scores, exact agreement, and quadratic weighted Cohen's kappa. The current rating sheet remains blank, so this notebook does not invent a human-quality result.

In [ ]:
if FINAL_READY:
    audit_dir = FINAL/'human_audit'; audit_dir.mkdir(exist_ok=True)
    pairs = [(report_A[i],report_fake_B[i],'A2B') for i in range(15)]
    pairs += [(report_B[i],report_fake_A[i],'B2A') for i in range(15)]
    random.Random(CONFIG['seed']+1000).shuffle(pairs)
    rating_rows,mapping = [],[]
    for i,(source,translated,direction) in enumerate(pairs):
        sample_id = f'sample_{i+1:02d}'
        canvas = PILImage.new('RGB',(512,256))
        canvas.paste(PILImage.fromarray(load_rgb(source,256)),(0,0))
        canvas.paste(PILImage.fromarray(load_rgb(translated,256)),(256,0))
        canvas.save(audit_dir/(sample_id+'.png'))
        rating_rows.append({'sample_id':sample_id,**{f'rater{r}_{c}':None for r in [1,2] for c in ['style','content','artifacts']}})
        mapping.append({'sample_id':sample_id,'direction':direction,'source':source.name,'translation':translated.name})
    audit_path = audit_dir/'ratings.csv'
    if not audit_path.exists(): pd.DataFrame(rating_rows).to_csv(audit_path,index=False)
    json_atomic(audit_dir/'private_mapping.json',mapping)
    with zipfile.ZipFile(FINAL/'human_audit_blinded.zip','w',zipfile.ZIP_STORED) as z:
        for p in sorted(audit_dir.glob('sample_*.png')): z.write(p,arcname=p.name)
        z.write(audit_path,arcname='ratings.csv')
    print('Two raters must independently fill:',audit_path)


In [ ]:
human_audit = {'status':'PENDING_TWO_RATERS'}
if FINAL_READY:
    aud = pd.read_csv(audit_path)
    expected = [f'sample_{i+1:02d}' for i in range(30)]
    assert aud.sample_id.tolist() == expected, 'Do not remove/reorder audit rows.'
    score_cols = [f'rater{r}_{c}' for r in [1,2] for c in ['style','content','artifacts']]
    if aud[score_cols].notna().all().all():
        scores = aud[score_cols].to_numpy(float)
        assert ((scores >= 1)&(scores <= 5)&(scores == scores.astype(int))).all(), 'Use integer scores 1–5.'
        human_audit = {'status':'COMPLETE'}
        for criterion in ['style','content','artifacts']:
            a,b = aud[f'rater1_{criterion}'],aud[f'rater2_{criterion}']
            kappa = cohen_kappa_score(a,b,weights='quadratic')
            human_audit[criterion+'_mean'] = float(np.mean([a.mean(),b.mean()]))
            human_audit[criterion+'_agreement_percent'] = float((a == b).mean()*100)
            human_audit[criterion+'_weighted_kappa'] = float(kappa) if np.isfinite(kappa) else None
    json_atomic(FINAL/'human_audit_summary.json',human_audit)
    print(json.dumps(human_audit,indent=2))


## 12. Build the submission CSV

The submission cell reads the organizer's template, preserves its `ID`, and replaces only the measured `FID` and `MiFID` fields. It checks the required columns and refuses to guess an identifier or scoring convention.

The saved epoch-190 submission row contains FID 107.21183928701066 and MiFID 0.41853196918964386. Their mean is 53.81518562810015. Local diagnostic metrics remain separate from this submitted row.

In [ ]:
SUBMISSION_READY = False
if FINAL_READY:
    if not TEMPLATE_PATH.exists():
        print('Submission BLOCKED: missing organizer template with the exact ID.')
    elif not ORGANIZER_METRIC_RULES_CONFIRMED:
        print('Submission BLOCKED: confirm the organizer preprocessing/MiFID rule first.')
    else:
        template = pd.read_csv(TEMPLATE_PATH,dtype=str,keep_default_na=False)
        assert len(template) == 1 and set(template.columns) == {'ID','FID','MiFID'}, 'Unexpected organizer schema; do not guess.'
        assert template.ID.str.strip().ne('').all(), 'Organizer template ID cannot be blank.'
        original_id = template['ID'].copy()
        template['FID'] = f'{final_FID:.9f}'
        template['MiFID'] = f"{mifid_result['MiFID']:.9f}"
        assert template['ID'].equals(original_id)
        template.to_csv(FINAL/'submission.csv',index=False)
        loaded = pd.read_csv(FINAL/'submission.csv',dtype=str,keep_default_na=False)
        assert loaded.ID.equals(original_id) and loaded.columns.tolist() == template.columns.tolist()
        SUBMISSION_READY = True
        print('Upload ONLY this CSV to Kaggle after review:',FINAL/'submission.csv')
        display(loaded)


## 13. Record the leaderboard result

Leaderboard fields should be filled only after the website accepts and scores the submission. A locally computed value is not a substitute for an accepted result.

For the recorded submission, the latest reported leaderboard score is **-53.8151**. Rank and public/private split are left blank because they were not supplied. Keeping those fields explicit avoids turning an unknown value into a claim.

In [ ]:
KAGGLE_PUBLIC_SCORE = None
KAGGLE_PRIVATE_SCORE = None
KAGGLE_RANK = None
KAGGLE_SUBMISSION_ID = None


## 14. Consolidated report and artifact checks

This section writes a machine-readable summary of the configuration, selected epoch, directional metrics, final-epoch losses, throughput, memory, parameter counts, hashes, audit status, and submission status.

Artifact checks confirm that training completed, epoch 190 was selected, all 7,038 photo outputs were generated, manifests and logs exist, and the human audit or leaderboard fields are not marked complete unless supporting evidence is present.

In [ ]:
if FINAL_READY:
    manifest = json.loads((RUN_DIR/'manifest.json').read_text())
    history = pd.DataFrame(progress['history'])
    last = history.iloc[-1].to_dict() if len(history) else {}
    events = [json.loads(line) for line in (RUN_DIR/'raw_events.jsonl').read_text().splitlines() if line.strip()]
    nonfinite_events = sum(row['event'] == 'nonfinite_failure' for row in events)
    report = {'experiment':CONFIG['run_name'],'config':CONFIG,'selected_epoch':selected['selected_epoch'],
              'training_completed_epochs':progress['epoch'],'training_completed_steps':progress['completed_steps'],
              'full_photo_FID':final_FID,'local_MiFID':mifid_result,'directional_metrics':report_rows,
              'training_final_epoch_losses':last,'gradient_norm_max':float(history.gradient_norm_max.max()),
              'generator_gradient_norm_mean':float(history.G_grad.mean()),
              'discriminator_gradient_norm_mean':float(history.D_grad.mean()),
              'nan_inf_loss_count':nonfinite_events,'training_update_time_sec':progress['training_seconds'],
              'training_session_wall_time_sec':progress['total_seconds'],
              'training_pairs_per_sec':progress['completed_steps']/max(progress['training_seconds'],1e-12),
              'real_input_images_per_sec':2*progress['completed_steps']/max(progress['training_seconds'],1e-12),
              'peak_memory_mb':progress['peak_memory_mb'],'generator_parameter_count':manifest['generator_parameters'],
              'discriminator_parameter_count':manifest['discriminator_parameters'],'hardware':manifest['GPU'],
              'human_audit':human_audit,'kaggle_public_score':KAGGLE_PUBLIC_SCORE,
              'kaggle_private_score':KAGGLE_PRIVATE_SCORE,'kaggle_rank':KAGGLE_RANK,'kaggle_submission_id':KAGGLE_SUBMISSION_ID,
              'stats_sha256':STATS_HASH,'data_sha256':DATA_HASH,'submission_ready':SUBMISSION_READY}
    json_atomic(FINAL/'metrics.json',report)
    # A single flattened CSV contains every report metric; nested config/details remain in JSON.
    rows = []
    def flatten(prefix,value):
        if isinstance(value,dict):
            for key,item in value.items(): flatten(prefix+'.'+key if prefix else key,item)
        elif isinstance(value,list):
            for index,item in enumerate(value): flatten(prefix+f'[{index}]',item)
        else: rows.append({'metric':prefix,'value':value})
    flatten('',report)
    pd.DataFrame(rows).to_csv(FINAL/'full_metrics_report.csv',index=False)
    checks = {'protocol_passed':PROTOCOL_OK,'training_complete':completed,'best_checkpoint':(CKPT/'best_fid.pt').exists(),
              'all_photo_outputs':len(all_monet_outputs),'expected_photos':len(photo_files),
              'raw_log':(RUN_DIR/'raw_events.jsonl').exists(),'manifest':(RUN_DIR/'manifest.json').exists(),
              'human_audit_complete':human_audit['status']=='COMPLETE','kaggle_submission_ready':SUBMISSION_READY,
              'kaggle_result_recorded':KAGGLE_PUBLIC_SCORE is not None and KAGGLE_RANK is not None}
    json_atomic(FINAL/'artifact_checks.json',checks)
    print(json.dumps(checks,indent=2))
    print('Preserve outputs and checkpoints from:',RUN_DIR)


## 15. Analysis and limitations

The recorded run improved fixed-subset FID through epoch 190 and then slightly reversed at epoch 200, so selecting the best measured checkpoint was better than automatically using the last epoch. Gradients remained finite and no NaN/Inf loss events were recorded, although one large gradient maximum means stability should still be discussed cautiously.

The directional metrics show a tradeoff: Monet-to-photo has higher generative precision but much lower recall, while photo-to-Monet covers more of the target feature space but has lower precision and greater perceptual change. The metrics use finite samples and pretrained feature models, so they do not fully describe human visual quality.

The fixed grid also shows concrete limitations. Some photo-to-Monet outputs preserve unwanted source overlays, retain distorted geometry, or introduce repeated texture and color casts. Some Monet-to-photo outputs remain soft and painterly rather than fully photographic. These observations are documented with filenames in `failure_analysis.md`.

Comparisons with a teammate should use the same image resolution, feature extractor, preprocessing, and sample counts. Scores from a different protocol should be labeled separately rather than treated as a direct model ranking.

## 16. Completion status

- [x] Verified both image domains and recorded their counts and hashes.
- [x] Trained two generators and two discriminators from random initialization.
- [x] Completed 200 epochs and selected epoch 190 by fixed-subset FID.
- [x] Saved loss, gradient, learning-rate, and checkpoint-FID curves.
- [x] Reported both-direction FID, KID, precision/recall, cycle L1, LPIPS, and content cosine.
- [x] Recorded parameters, hardware, runtime, throughput, peak memory, and zero non-finite losses.
- [x] Generated all 7,038 photo-to-Monet outputs and an inference manifest.
- [x] Generated the submission CSV and recorded the latest reported score of -53.8151.
- [ ] Complete the 30-sample audit with two independent human raters and report agreement.
- [ ] Complete the final teammate comparison and joint report review.

## References

- Zhu et al., *Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks* (ICCV 2017): https://arxiv.org/abs/1703.10593
- CycleGAN authors' reference implementation: https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix
- PyTorch automatic mixed precision examples: https://docs.pytorch.org/docs/stable/notes/amp_examples.html

AI assistance was used for code review, debugging, and drafting documentation. The saved measurements come from the run artifacts, and the student remains responsible for understanding and explaining the implementation and results.